## Classical 2D/3D Plotly Plots

In [ ]:
# -------- Imports --------
import sys
import os
import numpy as np
import scipy.sparse as sp
from _utility import *
from _fractures import *
from _initial_conditions import *
from _plotly_plots import *
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

np.random.seed(0)

If you want try out your own grid parameters, uncommment the code in red and set your own parameters. Do not change the 4th line that defines `dx, dy, dz`, as these values are purely determined from other set values.

In [ ]:
# -- Grid parameters --

# -- Lower/Upper Bound, Number of Points, Separation Between Points --

xmin,ymin,zmin,xmax,ymax,zmax,Nx,Ny,Nz,dx,dy,dz = get_grid_parameters()

# manually setting if you want to try other things

'''
xmin,ymin,zmin = -1,-1,-1
xmax,ymax,zmax = 1,1,1
Nx, Ny, Nz = 10, 10, 10
dx, dy, dz = (xmax-xmin)/(Nx-1), (ymax-ymin)/(Ny-1), (zmax-zmin)/(Nz-1)  ## NEVER CHANGE THIS LINE
'''


# -- Staggered Grid Points for each wavefield component of seismic wave equation w = [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

N_main,N_vx,N_vy,N_vz,N_sxy,N_sxz,N_syz,N_vel,N_stress,psi_len = get_grid_size(Nx,Ny,Nz)

Only change `file_label_2`, which controls the initial condition (Ricker or Gaussian), and `file_label_5`, which controls fracture geometry.

In [ ]:
## Configure which physical scenario to run for the notebook, times will be determined later
#Decide if quantum result will be calculated (takes a long time to run, don't run with over 6x6x6 case
calc_quantum = False


#If quantum is true, calculate quantum and classical, if false just calculate classical
file_label_1 = 'classical'
if calc_quantum:
    file_label_1='quantum'

#initial conditions: ricker_IC_1 or gaussian_IC_1
file_label_2 = 'ricker_IC_1'

#Use grid size to create this label 
file_label_3 = str(Nx)+'x'+str(Ny)+'x'+str(Nz)

#Use physical domain for this label

file_label_4 = str(xmin)+'to'+str(xmax)

#Fracture geometry 
# options: two_perp_xy_yz, no_fracture, one_xy, one_yz
file_label_5 = 'two_perp_xy_yz'

In [ ]:
# Important: The following implementation of the material parameters is not scalable. This code is only for demonstration purposes and assumes oracle access. 
# In a real-world scenario, the material parameters and the Hamiltonian need to be sparsly constructed on the QC itself.
# Furthermore, this code uses direct matrix exponentiation, which is inefficient for large matrices. Other integration methods should be used (e.g. Trotter-Suzuki, Qubitization).


# Density model and S matrix (3D with one horizontal fracture)

#Set fracture thickness
fracture_thickness = 0
#Set density and compliance matrix according to fracture geometry specified earlier
if file_label_5 == 'two_perp_xy_yz':
    rho_model, S_matrix = two_perpendicular_fractures(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
elif file_label_5 == 'no_fracture':
    rho_model, S_matrix = no_fractures(Nx,Ny,Nz,dx,dy,dz)
elif file_label_5 == 'one_xy':
    rho_model, S_matrix = one_horizontal_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
elif file_label_5 == 'one_yz':
    rho_model, S_matrix = one_vertical_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
else:
    rho_model, S_matrix = no_fractures(Nx,Ny,Nz,dx,dy,dz)

# -------- Simulation (3D Elastic) (Oracle Access) --------
(H, A, _, B_sqrt, B_inv, _) = FD_solver_3D_elastic_quantum(Nx, Ny, Nz, dx, dy, dz, rho_model, S_matrix)

H_pauli = SparsePauliOp.from_operator(Operator(H.toarray())) # Expensive conversion
print("Hamiltonian shape: ", H.shape)
print("Hamiltonian NNZ-Ratio: ", H.nnz/H.shape[0]**2)
print("Pauli Terms (inefficient representation): ", len(H_pauli))

# Number of grid points
print("Number of grid points: ", psi_len)

# Number of qubits
n = (H.shape[0]-1).bit_length()
print("Number of qubits (for wave field): ", n)

In [ ]:
# -- Initial Conditions (3D Elastic) --
# State vector: [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

#phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
if file_label_2 == 'ricker_IC_1':
    phi_0 = ricker_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
elif file_label_2 == 'gaussian_IC_1':
    phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
else:
    phi_0 = ricker_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
    
# Pad the initial conditions with zeros to match Hamiltonian size (power of 2)
phi_0 = np.concatenate([phi_0, np.zeros(H.shape[0] - psi_len)])
# Normalize the initial state and transform it to the energy basis
psi_0 = B_sqrt @ phi_0
norm = np.linalg.norm(psi_0)
psi_0 /= norm

# Number of non-zero initial state values
psi_0_nnz = np.sum(psi_0 != 0)
print('Initial state NNZ-Ratio:', psi_0_nnz/psi_len)

In [ ]:
# Time Integration (Has classical integration errors (DOP853))
def get_classical_state_vector(t):
    #phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='DOP853').y.T[-1] 
    phi_t = solve_ivp(lambda tau, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='RK45').y.T[-1] 
    psi_t_classical = B_sqrt @ phi_t
    return psi_t_classical

Time ranges include `[5e-6,1e-4], [5e-5,1e-3],[.005,.1],[.05,1],[.25,5],[.5,10]`

I found that only the first time range makes sense, as the other time ranges get chaotic too quickly.

The `np.linspace` below is structured as `np.linspace(t_start, t_end, n_timesteps)`

In [ ]:
history = [psi_0]
for t in np.linspace(5e-6,5e-4,100):
    history.append(get_classical_state_vector(t))

## 2D Plots

plotting functions come from the `_plotly_plots.py` file

In [ ]:
for i in range(Nz):
    fig = plot_time_slider_plotly(k_fixed=i, history=history, 
                               xmin=xmin, ymin=ymin, zmin=zmin, xmax=xmax, ymax=ymax, zmax=zmax, 
                               Nx=Nx, Ny=Ny, Nz=Nz, dx=dx, dy=dy, dz=dz)
    fig.show()

## 3D Plots

3D velocity plot:

In [ ]:
fig = plot_3d_velocity_plotly(history, xmin, ymin, zmin, xmax, ymax, zmax, Nx, Ny, Nz)
fig.show()

3D velocity plot with vector lengths scaled so smaller length vectors are more visible:

the `logboost` parameter controls how much the boost is.

In [ ]:
fig = plot_3d_velocity_plotly_log(history, xmin, ymin, zmin, xmax, ymax, zmax, Nx, Ny, Nz, log_boost=100)
fig.show()